In [1]:
import os
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import pyarrow as pa
import polars as pl
from matplotlib import pyplot as plt
from deltalake import DeltaTable, write_deltalake

In [2]:
sys.path.append(str("/root/capsule/src/"))

In [3]:
from connects_common_connectivity.models import (
    DataSet,
    Modality,
    DataItem,
    DataItemDataSetAssociation,
    Unit,
    CellFeatureDefinition,
    CellFeatureSet,
)

from connects_common_connectivity.arrow_utils import (
    build_arrow_schema,
    models_to_table,
    attach_linkml_metadata,
    build_cell_feature_matrix_schema,
)


# VISp inhib morph features

In [4]:
units_df = pd.read_csv("/data/visp-features-and-mapping/inh_visp_patchseq_morph_feature_definitions.csv")

In [5]:
units_df

,id,description,unit,data_type,range_min,range_max
0,axon_bias_x,Difference in axon extent in the x-dimension (...,MICRONS_LENGTH,<f4,0.0,NaN
1,axon_bias_y,Difference in axon extent in the y-dimension (...,MICRONS_LENGTH,<f4,NaN,NaN
2,axon_depth_pc_0,First principal component of PCA performed on ...,NONE,<f4,NaN,NaN
3,axon_depth_pc_1,Second principal component of PCA performed on...,NONE,<f4,NaN,NaN
4,axon_depth_pc_2,Third principal component of PCA performed on ...,NONE,<f4,NaN,NaN
5,axon_depth_pc_3,Fourth principal component of PCA performed on...,NONE,<f4,NaN,NaN
6,axon_depth_pc_4,Fifth principal component of PCA performed on ...,NONE,<f4,NaN,NaN
7,axon_depth_pc_5,Sixth principal component of PCA performed on ...,NONE,<f4,NaN,NaN
8,axon_emd_with_basal_dendrite,Earth mover’s distance metric calculated using...,ARBITRARY_UNIT,<f4,0.0,NaN
9,axon_exit_distance,The path distance from the soma surface to the...,MICRONS_LENGTH,<f4,0.0,NaN


In [6]:
ft_defs = []
for idx, row in units_df.iterrows():
    ft_def = CellFeatureDefinition(
        id=str(row['id']),
        description=str(row['description']),
        unit=str(row['unit']),
        data_type=str(row['data_type']),
        range_min=float(row['range_min']),
        range_max=float(row['range_max']),
    )
    ft_defs.append(ft_def)


In [7]:
schema = build_arrow_schema(CellFeatureDefinition)
table = models_to_table(ft_defs, schema)
table = attach_linkml_metadata(table, linkml_class="CellFeatureDefinition")  # version auto-populated

In [8]:
PATH = "../results/cellfeaturedefinition/visp_inh"
write_deltalake(PATH, table, mode="append")

In [9]:
cell_feature_set = CellFeatureSet(
    id='inh_visp_morph_features',
    description='Cell features used for analysis and MET-type definition for inhibitory cortical Patch-seq neurons',
    feature_definition_ids=[fd.id for fd in ft_defs],
    extraction_method='Computed via https://github.com/AllenInstitute/skeleton_keys.'
)

In [10]:
schema = build_arrow_schema(CellFeatureSet)
table = models_to_table([cell_feature_set], schema)
table = attach_linkml_metadata(table, linkml_class="CellFeatureSet")  # version auto-populated

PATH = "../results/cellfeatureset/visp_inh"
write_deltalake(PATH, table, mode="append")

In [11]:
schema = build_cell_feature_matrix_schema(cell_feature_set, ft_defs, cell_index_column="id")

In [12]:
data_feature_matrix = pd.read_csv("/data/visp-features-and-mapping/inh_ivscc_features_wide_unnormalized.csv")

In [13]:
data_feature_matrix

,specimen_id,axon_bias_x,axon_bias_y,axon_depth_pc_0,axon_depth_pc_1,axon_depth_pc_2,axon_depth_pc_3,axon_depth_pc_4,axon_depth_pc_5,axon_emd_with_basal_dendrite,...,basal_dendrite_mean_diameter,basal_dendrite_num_branches,basal_dendrite_soma_percentile_x,basal_dendrite_soma_percentile_y,basal_dendrite_stem_exit_down,basal_dendrite_stem_exit_side,basal_dendrite_stem_exit_up,basal_dendrite_total_length,basal_dendrite_total_surface_area,soma_aligned_dist_from_pia
0,601506507,180.833198,-249.830744,-255.225090,23.857132,-264.429951,-299.517754,-400.345397,28.557966,22.687988,...,0.918386,22.0,0.137298,0.522513,0.000000,0.500000,0.500000,2518.295630,7207.459813,357.159826
1,601790961,25.481124,434.251083,-216.809886,-153.378458,-303.881835,-117.440692,241.134085,-7.542952,39.412388,...,0.870701,42.0,0.480986,0.462676,0.000000,0.666667,0.333333,4256.093901,11691.149209,663.103005
2,601803754,42.650596,104.697843,1157.152004,3052.272959,-16.743603,996.874807,318.205713,-209.400944,16.735521,...,0.882682,62.0,0.460057,0.333470,0.125000,0.625000,0.250000,4108.235412,11384.542643,170.365060
3,601808698,42.200019,342.307240,277.368079,-594.419412,-899.916702,833.364479,-405.968840,475.309877,10.474801,...,0.729696,42.0,0.316378,0.047513,0.000000,1.000000,0.000000,2309.168299,5284.628374,460.737871
4,601810307,89.759903,239.040891,1201.597360,-121.156464,890.413919,-553.118489,130.516992,254.212458,12.408631,...,0.834698,48.0,0.454165,0.198232,0.333333,0.666667,0.000000,3226.120636,8470.180595,271.649192
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
515,992386952,16.392923,-109.526470,414.096515,9.791075,90.525572,-78.477266,-298.057252,-51.636147,24.289362,...,0.721978,27.0,0.241616,0.198933,0.000000,1.000000,0.000000,2307.122538,5088.249125,103.263504
516,992830261,93.658339,0.897162,-870.550429,-27.656963,517.408296,209.947128,-119.900792,112.955596,21.176288,...,0.669456,52.0,0.381341,0.383921,0.333333,0.666667,0.000000,3482.194884,7312.632229,647.564011
517,993243528,59.664688,-291.172271,157.466482,114.334986,115.102503,-351.831818,-321.849646,54.609863,18.277459,...,0.710571,47.0,0.395349,0.123575,0.200000,0.600000,0.200000,3264.027546,7252.320477,131.319709
518,993245688,79.241519,-252.564319,2101.276636,-1077.178481,853.407032,21.100093,458.230008,-67.524807,16.590548,...,0.852397,55.0,0.463691,0.219187,0.200000,0.800000,0.000000,4241.089015,11243.586458,207.960281


In [14]:
df = data_feature_matrix.drop("specimen_id", axis=1)
df['project_id'] = 'visp_inh_patchseq'
df['feature_set_id'] = 'inh_visp_morph_features'

df['id'] = data_feature_matrix['specimen_id'].astype('string')

In [15]:
# go through and cast the types of each column according to the cell feature definitions
for cfd in ft_defs:
    col = cfd.id
    df[col] = df[col].astype(np.dtype(cfd.data_type))


In [16]:
table = pa.Table.from_pandas(df, schema=schema, preserve_index=False)

In [17]:
PATH = "../results/cellfeatures/inh_morph_features/"
write_deltalake(PATH, table, mode="append", partition_by=["project_id", "feature_set_id"])

In [18]:
feat_df = pl.read_delta(PATH)

In [19]:
feat_df

id,axon_bias_x,axon_bias_y,axon_depth_pc_0,axon_depth_pc_1,axon_depth_pc_2,axon_depth_pc_3,axon_depth_pc_4,axon_depth_pc_5,axon_emd_with_basal_dendrite,axon_exit_distance,axon_exit_theta,axon_extent_x,axon_extent_y,axon_frac_above_basal_dendrite,axon_frac_below_basal_dendrite,axon_frac_intersect_basal_dendrite,axon_max_branch_order,axon_max_euclidean_distance,axon_max_path_distance,axon_mean_contraction,axon_num_branches,axon_soma_percentile_x,axon_soma_percentile_y,axon_total_length,basal_dendrite_bias_x,basal_dendrite_bias_y,basal_dendrite_calculate_number_of_stems,basal_dendrite_extent_x,basal_dendrite_extent_y,basal_dendrite_frac_above_axon,basal_dendrite_frac_below_axon,basal_dendrite_frac_intersect_axon,basal_dendrite_max_branch_order,basal_dendrite_max_euclidean_distance,basal_dendrite_max_path_distance,basal_dendrite_mean_contraction,basal_dendrite_mean_diameter,basal_dendrite_num_branches,basal_dendrite_soma_percentile_x,basal_dendrite_soma_percentile_y,basal_dendrite_stem_exit_down,basal_dendrite_stem_exit_side,basal_dendrite_stem_exit_up,basal_dendrite_total_length,basal_dendrite_total_surface_area,soma_aligned_dist_from_pia,project_id,feature_set_id
str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i32,f32,f32,f32,i32,f32,f32,f32,f32,f32,i32,f32,f32,f32,f32,f32,i32,f32,f32,f32,f32,i32,f32,f32,f32,f32,f32,f32,f32,f32,str,str
"""601506507""",180.833191,-249.83075,-255.225098,23.857132,-264.429962,-299.517761,-400.345398,28.557966,22.687988,21.366434,0.753477,393.747833,364.079346,0.0,0.0,1.0,12,452.063904,794.445679,0.79993,39,0.42234,0.872872,3477.662354,124.994995,106.454025,2,184.329514,677.569275,0.344419,0.005637,0.649944,6,411.633698,523.7005,0.788084,0.918386,22,0.137298,0.522512,0.0,0.5,0.5,2518.295654,7207.459961,357.159821,"""visp_inh_patchseq""","""inh_visp_morph_features"""
"""601790961""",25.481123,434.251068,-216.809891,-153.378464,-303.881836,-117.440689,241.134079,-7.542952,39.412388,23.363827,0.413627,436.951447,718.174683,0.463688,0.0,0.536312,28,583.322144,1053.978027,0.864694,289,0.475137,0.080489,14777.348633,36.594662,-45.43544,6,313.066193,407.682129,0.0,0.01831,0.98169,8,272.892242,322.548553,0.851932,0.870701,42,0.480986,0.462676,0.0,0.666667,0.333333,4256.09375,11691.149414,663.103027,"""visp_inh_patchseq""","""inh_visp_morph_features"""
"""601803754""",42.650597,104.697845,1157.151978,3052.272949,-16.743603,996.874817,318.205719,-209.40094,16.735521,15.924714,0.43383,569.871033,229.078171,0.513019,0.0,0.486981,30,330.553345,861.607483,0.750487,691,0.486983,0.008414,30913.128906,19.051867,16.631531,8,234.991699,185.685074,0.0,0.017616,0.982384,6,249.564148,281.814117,0.776152,0.882682,62,0.460057,0.33347,0.125,0.625,0.25,4108.235352,11384.542969,170.365067,"""visp_inh_patchseq""","""inh_visp_morph_features"""
"""601808698""",42.20002,342.307251,277.368073,-594.419434,-899.916687,833.364502,-405.968842,475.309875,10.474801,0.0,0.193495,423.536346,588.806213,0.177495,0.002972,0.819533,28,492.171387,862.05603,0.844058,483,0.427856,0.018089,15565.37207,66.103867,99.791977,4,365.984375,229.163132,0.0,0.0,1.0,7,217.113831,261.844452,0.891588,0.729696,42,0.316378,0.047513,0.0,1.0,0.0,2309.168213,5284.628418,460.737885,"""visp_inh_patchseq""","""inh_visp_morph_features"""
"""601810307""",89.759903,239.040894,1201.597412,-121.156464,890.41394,-553.118469,130.516998,254.212463,12.408631,28.647764,0.058997,375.706818,348.430023,0.112132,0.0,0.887868,36,351.588165,912.201355,0.842963,515,0.451587,0.023353,19300.591797,15.288828,115.287704,6,234.766724,297.006195,0.0,0.012564,0.987436,9,217.818817,269.330078,0.900498,0.834698,48,0.454165,0.198232,0.333333,0.666667,0.0,3226.120605,8470.180664,271.6492,"""visp_inh_patchseq""","""inh_visp_morph_features"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""992386952""",16.392923,-109.526474,414.096527,9.791076,90.525574,-78.477264,-298.057251,-51.636147,24.289362,25.889467,0.452497,493.8865

# VISp excitatory morph features

In [20]:
units_df = pd.read_csv("/data/visp-features-and-mapping/exc_visp_patchseq_morph_feature_definitions.csv")

In [21]:
units_df

,id,description,unit,data_type,range_min,range_max
0,apical_dendrite_bias_x,Difference in apical dendrite extent in the x-...,MICRONS_LENGTH,<f4,0.0,NaN
1,apical_dendrite_bias_y,Difference in apical dendrite extent in the y-...,MICRONS_LENGTH,<f4,NaN,NaN
2,apical_dendrite_depth_pc_0,First principal component of PCA performed on ...,NONE,<f4,NaN,NaN
3,apical_dendrite_depth_pc_1,Second principal component of PCA performed on...,NONE,<f4,NaN,NaN
4,apical_dendrite_depth_pc_2,Third principal component of PCA performed on ...,NONE,<f4,NaN,NaN
5,apical_dendrite_depth_pc_3,Fourth principal component of PCA performed on...,NONE,<f4,NaN,NaN
6,apical_dendrite_early_branch_path,Ratio of the maximum length of all the shorter...,RATIO,<f4,0.0,1.0
7,apical_dendrite_emd_with_basal_dendrite,Earth mover’s distance metric calculated using...,ARBITRARY_UNIT,<f4,0.0,NaN
8,apical_dendrite_extent_x,Total extent of the apical dendrite in the x-d...,MICRONS_LENGTH,<f4,0.0,NaN
9,apical_dendrite_extent_y,Total extent of the apical dendrite in the y-d...,MICRONS_LENGTH,<f4,0.0,NaN


In [22]:
ft_defs = []
for idx, row in units_df.iterrows():
    ft_def = CellFeatureDefinition(
        id=str(row['id']),
        description=str(row['description']),
        unit=str(row['unit']),
        data_type=str(row['data_type']),
        range_min=float(row['range_min']),
        range_max=float(row['range_max']),
    )
    ft_defs.append(ft_def)


In [23]:
schema = build_arrow_schema(CellFeatureDefinition)
table = models_to_table(ft_defs, schema)
table = attach_linkml_metadata(table, linkml_class="CellFeatureDefinition")  # version auto-populated

In [24]:
PATH = "../results/cellfeaturedefinition/visp_exc"
write_deltalake(PATH, table, mode="append")

In [25]:
cell_feature_set = CellFeatureSet(
    id='exc_visp_morph_features',
    description='Cell features used for analysis and MET-type definition for inhibitory cortical Patch-seq neurons',
    feature_definition_ids=[fd.id for fd in ft_defs],
    extraction_method='Computed via https://github.com/AllenInstitute/skeleton_keys.'
)

In [26]:
schema = build_arrow_schema(CellFeatureSet)
table = models_to_table([cell_feature_set], schema)
table = attach_linkml_metadata(table, linkml_class="CellFeatureSet")  # version auto-populated

PATH = "../results/cellfeatureset/visp_exc"
write_deltalake(PATH, table, mode="append")

In [27]:
schema = build_cell_feature_matrix_schema(cell_feature_set, ft_defs, cell_index_column="id")

In [28]:
# Patch-seq morph features
data_feature_matrix = pd.read_csv("/data/visp-features-and-mapping/morph_features_mMET_exc_wide_unnormalized.csv")

In [29]:
data_feature_matrix

,specimen_id,apical_dendrite_bias_x,apical_dendrite_bias_y,apical_dendrite_depth_pc_0,apical_dendrite_depth_pc_1,apical_dendrite_depth_pc_2,apical_dendrite_depth_pc_3,apical_dendrite_early_branch_path,apical_dendrite_emd_with_basal_dendrite,apical_dendrite_extent_x,...,basal_dendrite_mean_diameter,basal_dendrite_num_branches,basal_dendrite_soma_percentile_x,basal_dendrite_soma_percentile_y,basal_dendrite_stem_exit_down,basal_dendrite_stem_exit_side,basal_dendrite_stem_exit_up,basal_dendrite_total_length,basal_dendrite_total_surface_area,soma_aligned_dist_from_pia
0,601628311,147.076721,388.737332,-0.134846,-7.323705,-16.315251,2.834573,0.377581,63.287174,303.832247,...,0.738017,29.0,0.210607,0.859339,0.00,1.000000,0.000000,1842.712482,4251.727723,543.539902
1,603229579,117.918472,513.295648,181.612081,-49.761568,75.880212,18.681380,0.272368,49.580831,323.203125,...,0.678512,30.0,0.468498,0.894992,0.00,1.000000,0.000000,1559.948300,3349.079209,541.661529
2,603337985,74.871318,382.301002,-77.738239,-19.625268,-50.744753,-86.666445,0.384813,13.400124,254.446243,...,0.635489,28.0,0.223268,0.305389,0.25,0.500000,0.250000,1467.914882,2933.822544,458.134219
3,603481505,130.585618,424.042586,309.580620,-48.441029,104.774864,-43.013513,0.370210,45.136786,355.122409,...,0.836733,42.0,0.395452,0.840821,0.00,0.833333,0.166667,2744.146833,7285.432249,498.644308
4,603511210,15.832649,472.387609,-14.175712,70.002755,-47.532825,-23.029360,0.430200,58.767808,185.047517,...,1.043034,52.0,0.458953,0.697552,0.00,1.000000,0.000000,3741.880551,12434.022965,467.544831
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
384,1036259470,228.209968,320.424416,-147.339173,-42.495121,-8.410686,126.120305,0.713882,18.904243,367.508294,...,0.752672,21.0,0.427015,0.654684,0.20,0.600000,0.200000,1302.648057,3058.151016,902.189119
385,1039243792,47.861325,-19.775071,-153.537471,-11.667367,-42.131647,73.554038,0.675558,5.084050,327.860486,...,0.709122,6.0,0.296593,0.044088,0.50,0.500000,0.000000,602.683781,1340.126098,832.825477
386,1039273993,276.587273,495.311917,-116.335457,-47.326925,3.769657,45.892703,0.246732,42.765763,399.540591,...,0.820395,20.0,0.342886,0.735926,0.00,1.000000,0.000000,1384.082446,3628.216524,909.886408
387,1039503078,85.594834,226.283485,-132.778654,0.083811,-30.549618,12.146968,0.145168,17.924472,128.257085,...,0.819520,20.0,0.298099,0.163498,0.25,0.750000,0.000000,1663.345223,4280.269414,913.240052


In [30]:
ps_df = data_feature_matrix.drop("specimen_id", axis=1)
ps_df['project_id'] = 'visp_exc_patchseq'
ps_df['feature_set_id'] = 'exc_visp_morph_features'

ps_df['id'] = data_feature_matrix['specimen_id'].astype('string')

In [31]:
# WNM morph features
data_feature_matrix = pd.read_csv("/data/visp-features-and-mapping/RawFeaturesWide_ChamferCorr.csv", index_col=0)

In [32]:
data_feature_matrix

,swc_path,soma_aligned_dist_from_pia,basal_dendrite_max_euclidean_distance,apical_dendrite_num_branches,basal_dendrite_stem_exit_down,apical_dendrite_bias_y,apical_dendrite_extent_y,apical_dendrite_depth_pc_0,apical_dendrite_early_branch_path,apical_dendrite_mean_contraction,...,apical_dendrite_extent_x,apical_dendrite_std_moments_along_max_distance_projection,apical_dendrite_num_outer_bifurcations,apical_dendrite_max_path_distance,apical_dendrite_depth_pc_1,basal_dendrite_calculate_number_of_stems,basal_dendrite_mean_contraction,apical_dendrite_bias_x,basal_dendrite_bias_x,basal_dendrite_frac_below_apical_dendrite
0,17109_6201-X4328-Y6753_reg,755.648634,153.118741,13.112095,0.519306,51.220063,130.348098,-127.799230,0.812125,0.919582,...,92.771122,0.095100,0.017473,188.477520,-4.550537,2.025048,0.924001,116.965793,73.426861,0.038531
1,17109_6301-X4756-Y24516_reg,779.803826,158.644468,11.710442,0.346235,29.822118,187.891473,-125.915996,0.464351,0.942480,...,133.943725,0.047786,0.017473,298.755658,-2.846434,3.016802,0.954847,68.235786,20.734153,-0.005611
2,17109_6601-X4384-Y7436_reg,727.831495,218.932446,18.718725,0.000094,53.307739,162.011671,-130.399448,0.573624,0.931142,...,139.756745,0.170515,0.759907,229.434422,-9.068610,2.025048,0.928544,144.969883,46.419273,0.232868
3,17109_6601-X5417-Y25287_reg,753.796313,163.343819,18.718725,0.000094,-134.153298,160.406543,-134.813898,0.610748,0.945982,...,153.129434,0.140594,0.017473,296.258962,-6.212346,4.008555,0.914963,18.596118,13.985249,-0.005611
4,17109_6801-X7432-Y4405_reg,616.766200,210.691787,14.513755,0.259700,-152.226616,176.436476,-128.454254,0.393457,0.908202,...,149.504220,0.169069,0.304686,340.347681,-7.067314,4.008555,0.922157,175.735165,34.455671,-0.005611
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
341,220308_8021-X22273-Y14515_reg,896.101355,154.007736,31.333636,0.000094,426.478646,508.423961,-89.878750,0.301818,0.843487,...,171.156424,0.348784,0.927916,584.360151,-42.625012,3.016802,0.847473,27.566970,64.863695,0.052311
342,220309_5824-X3486-Y10261_reg,606.551544,340.180443,29.931981,0.000094,511.616625,712.083170,-29.732973,0.438712,0.873898,...,305.753978,0.206086,0.591899,659.719187,-44.856429,5.992063,0.851315,28.460744,58.614060,0.058337
343,220309_6744-X9496-Y18547_reg,273.802444,194.949396,31.333636,0.173165,246.234974,307.956904,54.324096,0.540375,0.890393,...,254.576710,0.263005,1.047120,417.626183,19.109618,5.992063,0.839626,2.705247,43.218620,0.153476
344,220315_6442-X7542-Y17434_reg,404.821656,163.812376,25.727008,0.000094,225.254902,479.609711,-64.358903,0.473110,0.903657,...,250.183733,0.251633,0.472695,455.991818,-11.938175,4.008555,0.884905,38.164678,23.977165,-0.005611


In [33]:
wnm_df = data_feature_matrix.drop(["swc_path"], axis=1)
wnm_df['project_id'] = 'visp_exc_wnm'
wnm_df['feature_set_id'] = 'exc_visp_morph_features'

wnm_df['id'] = data_feature_matrix['swc_path'].astype('string')

In [34]:
df = pd.concat([ps_df, wnm_df])

In [35]:
df

,apical_dendrite_bias_x,apical_dendrite_bias_y,apical_dendrite_depth_pc_0,apical_dendrite_depth_pc_1,apical_dendrite_depth_pc_2,apical_dendrite_depth_pc_3,apical_dendrite_early_branch_path,apical_dendrite_emd_with_basal_dendrite,apical_dendrite_extent_x,apical_dendrite_extent_y,...,basal_dendrite_soma_percentile_y,basal_dendrite_stem_exit_down,basal_dendrite_stem_exit_side,basal_dendrite_stem_exit_up,basal_dendrite_total_length,basal_dendrite_total_surface_area,soma_aligned_dist_from_pia,project_id,feature_set_id,id
0,147.076721,388.737332,-0.134846,-7.323705,-16.315251,2.834573,0.377581,63.287174,303.832247,430.002777,...,0.859339,0.000000,1.000000,0.000000,1842.712482,4251.727723,543.539902,visp_exc_patchseq,exc_visp_morph_features,601628311
1,117.918472,513.295648,181.612081,-49.761568,75.880212,18.681380,0.272368,49.580831,323.203125,680.002141,...,0.894992,0.000000,1.000000,0.000000,1559.948300,3349.079209,541.661529,visp_exc_patchseq,exc_visp_morph_features,603229579
2,74.871318,382.301002,-77.738239,-19.625268,-50.744753,-86.666445,0.384813,13.400124,254.446243,446.575015,...,0.305389,0.250000,0.500000,0.250000,1467.914882,2933.822544,458.134219,visp_exc_patchseq,exc_visp_morph_features,603337985
3,130.585618,424.042586,309.580620,-48.441029,104.774864,-43.013513,0.370210,45.136786,355.122409,644.199169,...,0.840821,0.000000,0.833333,0.166667,2744.146833,7285.432249,498.644308,visp_exc_patchseq,exc_visp_morph_features,603481505
4,15.832649,472.387609,-14.175712,70.002755,-47.532825,-23.029360,0.430200,58.767808,185.047517,501.410723,...,0.697552,0.000000,1.000000,0.000000,3741.880551,12434.022965,467.544831,visp_exc_patchseq,exc_visp_morph_features,603511210
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
341,27.566970,426.478646,-89.878750,-42.625012,7.690337,-8.018195,0.301818,44.870007,171.156424,508.423961,...,0.453674,0.000094,0.999812,0.001988,1088.549274,NaN,896.101355,visp_exc_wnm,exc_visp_morph_features,220308_8021-X22273-Y14515_reg
342,28.460744,511.616625,-29.732973,-44.856429,68.863926,-11.209417,0.438712,29.809223,305.753978,712.083170,...,0.763099,0.000094,0.999812,0.001988,2761.256656,NaN,606.551544,visp_exc_wnm,exc_visp_morph_features,220309_5824-X3486-Y10261_reg
343,2.705247,246.234974,54.324096,19.109618,-88.036493,45.803957,0.540375,27.151869,254.576710,307.956904,...,0.306175,0.173165,0.833121,0.001988,2497.913181,NaN,273.802444,visp_exc_wnm,exc_visp_morph_features,220309_6744-X9496-Y18547_reg
344,38.164678,225.254902,-64.358903,-11.938175,-25.288155,-58.882341,0.473110,14.331949,250.183733,479.609711,...,0.848069,0.000094,0.999812,0.001988,1535.059609,NaN,404.821656,visp_exc_wnm,exc_visp_morph_features,220315_6442-X7542-Y17434_reg


In [36]:
# go through and cast the types of each column according to the cell feature definitions
for cfd in ft_defs:
    col = cfd.id
    df[col] = df[col].astype(np.dtype(cfd.data_type))


In [37]:
table = pa.Table.from_pandas(df, schema=schema, preserve_index=False)

In [38]:
PATH = "../results/cellfeatures/exc_morph_features/"
write_deltalake(PATH, table, mode="append", partition_by=["project_id", "feature_set_id"])

In [39]:
feat_df = pl.read_delta(PATH)

In [40]:
feat_df

id,apical_dendrite_bias_x,apical_dendrite_bias_y,apical_dendrite_depth_pc_0,apical_dendrite_depth_pc_1,apical_dendrite_depth_pc_2,apical_dendrite_depth_pc_3,apical_dendrite_early_branch_path,apical_dendrite_emd_with_basal_dendrite,apical_dendrite_extent_x,apical_dendrite_extent_y,apical_dendrite_frac_above_basal_dendrite,apical_dendrite_frac_below_basal_dendrite,apical_dendrite_frac_intersect_basal_dendrite,apical_dendrite_max_branch_order,apical_dendrite_max_euclidean_distance,apical_dendrite_max_path_distance,apical_dendrite_mean_contraction,apical_dendrite_mean_diameter,apical_dendrite_mean_moments_along_max_distance_projection,apical_dendrite_num_branches,apical_dendrite_num_outer_bifurcations,apical_dendrite_soma_percentile_x,apical_dendrite_soma_percentile_y,apical_dendrite_std_moments_along_max_distance_projection,apical_dendrite_total_length,apical_dendrite_total_surface_area,axon_exit_distance,axon_exit_theta,basal_dendrite_bias_x,basal_dendrite_bias_y,basal_dendrite_calculate_number_of_stems,basal_dendrite_extent_x,basal_dendrite_extent_y,basal_dendrite_frac_above_apical_dendrite,basal_dendrite_frac_below_apical_dendrite,basal_dendrite_frac_intersect_apical_dendrite,basal_dendrite_max_branch_order,basal_dendrite_max_euclidean_distance,basal_dendrite_max_path_distance,basal_dendrite_mean_contraction,basal_dendrite_mean_diameter,basal_dendrite_num_branches,basal_dendrite_soma_percentile_x,basal_dendrite_soma_percentile_y,basal_dendrite_stem_exit_down,basal_dendrite_stem_exit_side,basal_dendrite_stem_exit_up,basal_dendrite_total_length,basal_dendrite_total_surface_area,soma_aligned_dist_from_pia,project_id,feature_set_id
str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i32,f32,f32,f32,f32,f32,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i32,f32,f32,f32,f32,f32,i32,f32,f32,f32,f32,i32,f32,f32,f32,f32,f32,f32,f32,f32,str,str
"""601628311""",147.076721,388.737335,-0.134846,-7.323705,-16.31525,2.834573,0.377581,63.287174,303.832245,430.002777,0.647374,0.0,0.352626,9,412.426788,535.364746,0.880161,0.903242,0.395395,21,0.778151,0.199797,0.022312,0.297191,1513.543823,4145.918457,0.0,0.676106,190.920303,-215.522324,3,298.811951,262.765289,0.0,0.477325,0.522675,8,381.776184,445.646149,0.886983,0.738017,29,0.210607,0.859339,0.0,1.0,0.0,1842.712524,4251.727539,543.539917,"""visp_exc_patchseq""","""exc_visp_morph_features"""
"""603229579""",117.918472,513.295654,181.612076,-49.761566,75.880211,18.681379,0.272368,49.58083,323.203125,680.002136,0.76549,0.0,0.23451,18,614.681091,779.244324,0.918641,0.765617,0.417079,63,1.113943,0.280784,0.083137,0.385682,3165.540771,7558.615234,0.0,0.566352,6.077639,-85.867607,4,219.374954,150.459167,0.0,0.033118,0.966882,5,146.587982,167.992691,0.898062,0.678512,30,0.468498,0.894992,0.0,1.0,0.0,1559.948242,3349.079102,541.661499,"""visp_exc_patchseq""","""exc_visp_morph_features"""
"""603337985""",74.871315,382.300995,-77.738235,-19.625267,-50.744755,-86.666443,0.384813,13.400125,254.446243,446.575012,0.248392,0.0,0.751608,7,436.830353,504.563934,0.85786,0.68569,0.154741,25,0.30103,0.242765,0.057074,0.239363,1463.44458,3153.132812,0.0,0.777599,19.367064,75.317406,4,192.531525,190.774384,0.0,0.0,1.0,5,146.894363,176.792923,0.850834,0.635489,28,0.223268,0.305389,0.25,0.5,0.25,1467.914917,2933.82251,458.134216,"""visp_exc_patchseq""","""exc_visp_morph_features"""
"""603481505""",130.585617,424.042572,309.580627,-48.441029,104.774864,-43.013512,0.37021,45.136787,355.122406,644.199158,0.597045,0.0,0.402955,20,564.46106,831.234009,0.81778,0.971846,0.382736,75,1.230449,0.243337,0.17613,0.342804,5530.317383,16620.470703,0.0,0.58749,3.566612,-62.815819,6,242.697159,149.831894,0.0,0.0,1.0,4,151.566345,196.850922,0.781844,0.836733,42,0.395452,0.840821,0.0,0.833333,0.166667,2744.146729,7285.432129,498.644318,"""visp_exc_patchseq""","""exc_visp_morph_features"""
"""603511210""",15.832649,472.387604,-14.175712,70.002754,-47.532825,-23.02936,0.4302,58.767807,185.047516,501.410736,0.908025,0.0,0.091975,7,510.1